# Gold Layer - Silver to Gold Aggregation (via DBT)

## 🌐 Service UIs (Click These!)
- **HDFS UI**: http://localhost:9870 | **File Browser**: http://localhost:9870/explorer.html#/lakehouse
- **Trino (Query)**: http://localhost:8083
- **Airflow (Orchestration)**: http://localhost:8090 (user: airflow, password: airflow)
- **Spark Master** (Docker): http://localhost:8082 | **Spark UI** (local session): http://localhost:4040
- **Grafana (Monitoring)**: http://localhost:3000 (user: admin, password: admin)
- **Prometheus (Metrics)**: http://localhost:9090
- **Schema Registry**: http://localhost:8081
- **PostgreSQL**: localhost:5432 (user: airflow, password: airflow, db: gold_layer)
- **Kafka Brokers**: localhost:19092, localhost:19093, localhost:19094

**📖 For detailed service info, see** [START_HERE.md](../START_HERE.md)

This notebook runs DBT to transform Silver → Gold layer:
- Reads from: `hdfs://namenode:9000/lakehouse/silver/silver_booking_state`
- Writes to:
  - **HDFS Iceberg**: `hdfs://namenode:9000/lakehouse/gold/gold_daily_kpis_v2` (Snappy Parquet)
  - **PostgreSQL**: `gold_layer.gold.gold_daily_kpis_postgres` table
- Aggregations: Daily KPIs by city (bookings, revenue, cancellation rate, etc.)

### Cross-Database Architecture:
1. **Iceberg Creation**: DBT runs `gold_daily_kpis_v2` with `--target local` (Spark/Iceberg)
2. **PostgreSQL Structure**: DBT creates empty table structure with `--target prod` (PostgreSQL)
3. **Data Transfer**: Python script (`transfer_to_postgres.py`) reads from Iceberg and populates PostgreSQL table

In [1]:
import sys
import os
import subprocess
sys.path.append('../')

# Install additional dependencies for cross-database transfer
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas>=2.0.0', 'sqlalchemy>=2.0.0', 'psycopg2-binary>=2.9.0', '-q'], check=False)

# Set Java 17 for PySpark 3.5.0 compatibility
# os.environ['JAVA_HOME'] = '/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home'
os.environ['JAVA_HOME'] = '/opt/java/openjdk'
import sys
from pyspark.sql import SparkSession
import pandas as pd
import matplotlib.pyplot as plt

## 1. Initialize Spark Session

In [2]:
import sys
import os
import subprocess

print("Python executable:", sys.executable)
print("JAVA_HOME:", os.environ.get('JAVA_HOME', 'NOT SET'))
print("\nRunning: which java")
result = subprocess.run(['which', 'java'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

print("\nRunning: java -version")
result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print(result.stderr)  # java -version writes to stderr

print("\nChecking if JAVA_HOME path exists:")
java_home = os.environ.get('JAVA_HOME')
if java_home:
    print(f"Path exists: {os.path.exists(java_home)}")
    print(f"Java bin exists: {os.path.exists(os.path.join(java_home, 'bin', 'java'))}")

Python executable: /usr/bin/python3
JAVA_HOME: /opt/java/openjdk

Running: which java
/opt/java/openjdk/bin/java



Running: java -version
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)


Checking if JAVA_HOME path exists:
Path exists: True
Java bin exists: True


In [3]:
# Stop any existing Spark session
try:
    spark.stop()
    print("Stopped existing Spark session")
except:
    pass

print(os.environ['JAVA_HOME'])
# Create Spark session with local filesystem (for notebook development)
spark = SparkSession.builder \
    .appName("Gold-Layer-DBT") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,org.apache.hadoop:hadoop-client:3.3.6") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "hdfs://namenode:9000/lakehouse") \
    .config("spark.sql.iceberg.compression-codec", "snappy") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.iceberg.write.format.default", "parquet") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

print("✅ Spark session initialized for DBT!")
print(f"   Version: {spark.version}")
print(f"   Warehouse: hdfs://namenode:9000/lakehouse")
print(f"   Compression: Snappy Parquet")
print(f"   Note: Using local filesystem for notebook development")

/opt/java/openjdk
:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-client added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-425b93f1-c9e1-4988-b7e6-97cbe12f4b7b;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.3 in central
	found org.apache.hadoop#hadoop-client;3.3.6 in central
	found org.apache.hadoop#hadoop-common;3.3.6 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-protobuf_3_7;1.1.1 in central
	found org.apache.hadoop#hadoop-annotations;3.3.6 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.1.1 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central
	found com.google.code.findbugs#jsr305;3.0.2 in centra

✅ Spark session initialized for DBT!
   Version: 3.5.0
   Warehouse: hdfs://namenode:9000/lakehouse
   Compression: Snappy Parquet
   Note: Using local filesystem for notebook development


## 2. Verify Silver Layer Data

In [4]:
print("📋 Silver Layer Tables:")
spark.sql("SHOW TABLES IN local.silver").show()

print("\n📊 Silver Booking State:")
silver_df = spark.table("local.silver.silver_booking_state")
print(f"Count: {silver_df.count()}")
silver_df.show(5)

📋 Silver Layer Tables:
+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|   silver|silver_booking_state|      false|
+---------+--------------------+-----------+


📊 Silver Booking State:
Count: 55
+----------+-------+--------+---------+-----+-------------------+-------------------+------+-----------+--------------------+--------------------+
|booking_id|user_id|hotel_id|   status|price|         created_at|         updated_at|  city|star_rating|bronze_processing_ts|     last_updated_ts|
+----------+-------+--------+---------+-----+-------------------+-------------------+------+-----------+--------------------+--------------------+
|        b1|     u1|      h1|confirmed|120.0|2025-01-01 10:05:00|2025-01-01 10:10:00|Tehran|          4|2026-02-21 16:52:...|2026-02-21 17:04:...|
|        b1|     u1|      h1|confirmed|120.0|2025-01-01 10:05:00|2025-01-01 10:10:00|Tehran|          4|2026-02-21 16:52:...|202

## 3. Run DBT Gold Layer Transformation (HDFS Iceberg)

In [5]:
# Change to DBT directory
os.chdir('../dbt')

# Use HDFS warehouse so dbt writes to same location as this notebook's Spark session
os.environ["LAKEHOUSE_WAREHOUSE"] = "hdfs://namenode:9000/lakehouse"

print("🚀 Running DBT Gold Layer transformation (HDFS Iceberg)...\n")
print("=" * 70)

# Run DBT for Gold layer (Iceberg only, not Postgres model)
result = subprocess.run(
    ['dbt', 'run', '--select', 'gold_daily_kpis_v2', '--profiles-dir', '.', '--profile', 'snapptrip', '--target', 'local'],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("\nWarnings/Errors:")
    print(result.stderr)

print("=" * 70)

if result.returncode == 0:
    print("\n✅ DBT Gold layer transformation (HDFS) completed successfully!")
else:
    print(f"\n❌ DBT run failed with exit code {result.returncode}")

# Return to notebooks directory
os.chdir('../notebooks')

🚀 Running DBT Gold Layer transformation (HDFS Iceberg)...

23:54:54  Running with dbt=1.7.4
23:54:54  Registered adapter: spark=1.7.1
23:54:54  Unable to do partial parsing because config vars, config profile, or config target have changed
23:54:54  Unable to do partial parsing because env vars used in profiles.yml have changed
23:54:54  Unable to do partial parsing because a project dependency has been added
23:54:55  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 2 unused configuration paths:
- models.snapptrip_gold.staging
- seeds.snapptrip_gold
23:54:55  Found 8 models, 57 tests, 3 sources, 0 exposures, 0 metrics, 898 macros, 0 groups, 0 semantic models
23:54:55  
:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
23:55:02  Concurrency: 4 threads (target='local')
23:55:02  
23:55:02  1 of 1 START sql incremental model 

## 4. Run DBT Data Quality Tests

In [6]:
os.chdir('../dbt')

print("🧪 Running DBT data quality tests for Gold layer...\n")
print("=" * 70)

# Run DBT tests
result = subprocess.run(
    ['dbt', 'test', '--select', 'gold_daily_kpis_v2', '--profiles-dir', '.', '--profile', 'snapptrip', '--target', 'local'],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("\nWarnings/Errors:")
    print(result.stderr)

print("=" * 70)

if result.returncode == 0:
    print("\n✅ All data quality tests passed!")
else:
    print(f"\n⚠️  Some tests failed (exit code {result.returncode})")

os.chdir('../notebooks')

🧪 Running DBT data quality tests for Gold layer...

23:55:09  Running with dbt=1.7.4
23:55:09  Registered adapter: spark=1.7.1
23:55:09  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 2 unused configuration paths:
- models.snapptrip_gold.staging
- seeds.snapptrip_gold
23:55:09  Found 8 models, 57 tests, 3 sources, 0 exposures, 0 metrics, 898 macros, 0 groups, 0 semantic models
23:55:09  
:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
23:55:15  Concurrency: 4 threads (target='local')
23:55:15  
23:55:15  1 of 18 START test dbt_utils_expression_is_true_gold_daily_kpis_v2_avg_booking_price___0  [RUN]
23:55:15  2 of 18 START test dbt_utils_expression_is_true_gold_daily_kpis_v2_booking_date_CURRENT_DATE_  [RUN]
23:55:15  3 of 18 START test dbt_utils_expression_is_true_gold_daily_kpis_v2_cancellation_rate__BETWEEN_0_AND_100 

## 5. Verify Gold Layer in HDFS

In [7]:
print("📋 Gold Layer Tables:")
try:
    spark.sql("SHOW TABLES IN local.gold").show()
    print("\n📊 Gold Daily KPIs:")
    gold_df = spark.table("local.gold.gold_daily_kpis_v2")
    print(f"Count: {gold_df.count()}")
    print("\nSchema:")
    gold_df.printSchema()
    print("\nSample data:")
    gold_df.show(20, truncate=False)
except Exception as e:
    err = str(e)
    if "NoSuchNamespaceException" in err or "Namespace does not exist" in err or "gold" in err:
        _wh = "hdfs://namenode:9000/lakehouse"
        try:
            _wh = spark.conf.get("spark.sql.catalog.local.warehouse", _wh)
        except Exception:
            pass
        print("❌ Gold namespace not found in catalog 'local' (warehouse:", _wh + ").")
        print("\n   To fix:")
        print("   1. Restart the kernel, then re-run from the top (so Spark uses HDFS warehouse).")
        print("   2. Ensure the Spark init cell has: spark.sql.catalog.local.warehouse = hdfs://namenode:9000/lakehouse")
        print("   3. Ensure dbt profile 'local' target uses the same warehouse; re-run the DBT Gold cell.")
        print("   4. Then re-run this verification cell.")
    else:
        raise

📋 Gold Layer Tables:
+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|     gold|gold_daily_kpis_v2|      false|
+---------+------------------+-----------+


📊 Gold Daily KPIs:
Count: 5

Schema:
root
 |-- booking_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- total_bookings: long (nullable = true)
 |-- confirmed_bookings: long (nullable = true)
 |-- cancelled_bookings: long (nullable = true)
 |-- pending_bookings: long (nullable = true)
 |-- cancellation_rate: double (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- avg_confirmed_price: double (nullable = true)
 |-- avg_booking_price: double (nullable = true)
 |-- min_price: double (nullable = true)
 |-- max_price: double (nullable = true)
 |-- avg_star_rating: double (nullable = true)
 |-- unique_customers: long (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- dbt_updated_at: timestamp (nullable

## 6. Write Gold Layer to PostgreSQL via DBT

In [8]:
# Install dbt-postgres 1.7.x (matches dbt-core 1.7.x; dbt-spark 1.7.1 requires ~=1.7.0)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'dbt-core==1.7.4', 'dbt-postgres==1.7.4', '-q'], check=False)

os.chdir('../dbt')

print("🚀 Transferring Gold Layer to PostgreSQL (Cross-Database)...\n")
print("=" * 70)

# Step 1: Create PostgreSQL table structure
print("Step 1: Creating PostgreSQL table structure...")
pg_result = subprocess.run(
    ['dbt', 'run', '--select', 'gold_daily_kpis_postgres', '--profiles-dir', '.', '--profile', 'snapptrip', '--target', 'prod'],
    capture_output=True,
    text=True
)

print(pg_result.stdout)
if pg_result.stderr:
    print("PostgreSQL table creation warnings/errors:")
    print(pg_result.stderr)

if pg_result.returncode != 0:
    print(f"❌ PostgreSQL table creation failed with exit code {pg_result.returncode}")
    os.chdir('../notebooks')
else:
    print("✅ PostgreSQL table structure created!")
    
    # Step 2: Transfer data from Iceberg to PostgreSQL using Python script
    print("\nStep 2: Running cross-database transfer script...")
    os.environ["LAKEHOUSE_WAREHOUSE"] = "hdfs://namenode:9000/lakehouse"
    os.environ["POSTGRES_HOST"] = "postgres"
    os.environ["POSTGRES_DB"] = "gold_layer"
    os.environ["POSTGRES_USER"] = "airflow"
    os.environ["POSTGRES_PASSWORD"] = "airflow"
    
    transfer_result = subprocess.run(
        [sys.executable, 'scripts/transfer_to_postgres.py'],
        capture_output=True,
        text=True
    )
    
    print(transfer_result.stdout)
    if transfer_result.stderr:
        print("Transfer warnings/errors:")
        print(transfer_result.stderr)
    
    if transfer_result.returncode == 0:
        print("✅ Data transferred successfully from Iceberg to PostgreSQL!")
    else:
        print(f"❌ Cross-database transfer failed with exit code {transfer_result.returncode}")

print("=" * 70)
os.chdir('../notebooks')

🚀 Transferring Gold Layer to PostgreSQL (Cross-Database)...

Step 1: Creating PostgreSQL table structure...
23:55:23  Running with dbt=1.7.4
23:55:23  Registered adapter: postgres=1.7.4
23:55:23  Unable to do partial parsing because config vars, config profile, or config target have changed
23:55:23  Unable to do partial parsing because env vars used in profiles.yml have changed
23:55:23  Unable to do partial parsing because a project dependency has been added
23:55:24  [WARNING]: Configuration paths exist in your dbt_project.yml file which do not apply to any resources.
There are 2 unused configuration paths:
- models.snapptrip_gold.staging
- seeds.snapptrip_gold
23:55:24  Found 8 models, 57 tests, 3 sources, 0 exposures, 0 metrics, 860 macros, 0 groups, 0 semantic models
23:55:24  
23:55:24  Concurrency: 4 threads (target='prod')
23:55:24  
23:55:24  1 of 1 START sql table model gold.gold_daily_kpis_postgres ..................... [RUN]
23:55:24  1 of 1 OK created sql table model gold

## 7. Verify PostgreSQL Data

In [9]:
import psycopg

# Connect to PostgreSQL
conn = psycopg.connect(
    host="postgres",
    port=5432,
    dbname="gold_layer",
    user="airflow",
    password="airflow"
)

print("✅ Connected to PostgreSQL")

# Query the Gold layer table
query = "SELECT * FROM gold.gold_daily_kpis_postgres ORDER BY booking_date DESC, city LIMIT 20"
pg_df = pd.read_sql(query, conn)

print(f"\n📊 PostgreSQL Gold Layer: {len(pg_df)} rows")
print(pg_df)

conn.close()

✅ Connected to PostgreSQL

📊 PostgreSQL Gold Layer: 5 rows
  booking_date     city  total_bookings  confirmed_bookings  \
0   2025-01-03  Isfahan               1                   0   
1   2025-01-03   Shiraz               1                   0   
2   2025-01-02   Shiraz               1                   0   
3   2025-01-02   Tehran               1                   1   
4   2025-01-01   Tehran               1                   1   

   cancelled_bookings  pending_bookings  cancellation_rate  total_revenue  \
0                   0                 1                0.0            0.0   
1                   1                 0              100.0            0.0   
2                   1                 0              100.0            0.0   
3                   0                 0                0.0         1650.0   
4                   0                 0                0.0         1320.0   

   avg_confirmed_price  avg_booking_price  min_price  max_price  \
0                  NaN          

/tmp/ipykernel_47201/4276223853.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pg_df = pd.read_sql(query, conn)


## 8. Check HDFS Storage for Gold Layer

In [10]:
print("📁 HDFS Gold Layer Directory & Data:")
print("\n" + "=" * 70)

# Use Spark to explore the Gold layer Iceberg data directly
try:
    # Show Iceberg table metadata and schema
    print("\n🔍 Gold Daily KPIs Table Schema:")
    spark.sql("DESCRIBE TABLE local.gold.gold_daily_kpis_v2").show()
    
    # Show sample data
    print("\n📊 Sample Gold Layer Data (First 10 records):")
    df_gold = spark.sql("SELECT * FROM local.gold.gold_daily_kpis_v2 ORDER BY booking_date DESC, city LIMIT 10")
    df_gold.show(truncate=False)
    
    # Show data summary statistics
    print("\n📈 Data Summary:")
    row_count = spark.sql("SELECT COUNT(*) as total_records FROM local.gold.gold_daily_kpis_v2").collect()[0]['total_records']
    date_range = spark.sql("""
        SELECT 
            MIN(booking_date) as earliest_date,
            MAX(booking_date) as latest_date,
            COUNT(DISTINCT city) as unique_cities
        FROM local.gold.gold_daily_kpis_v2
    """).collect()[0]
    
    print(f"Total Records: {row_count}")
    print(f"Date Range: {date_range['earliest_date']} to {date_range['latest_date']}")
    print(f"Unique Cities: {date_range['unique_cities']}")
    
    # Show table location information
    print("\n🗂️ Table Location Info:")
    table_info = spark.sql("SHOW TBLPROPERTIES local.gold.gold_daily_kpis_v2")
    table_info.filter("key LIKE '%location%' OR key LIKE '%path%'").show(truncate=False)
    
except Exception as e:
    print(f"Error accessing Iceberg table: {e}")
    print("\nTrying alternative approach...")
    
    # Alternative: Try to access via file system if Iceberg fails
    try:
        hadoop_files = spark.sql("SELECT input_file_name() as file_path FROM parquet.`hdfs://namenode:9000/lakehouse/gold/gold_daily_kpis_v2/data/*.parquet` LIMIT 5")
        print("\n📁 Gold Layer Files:")
        hadoop_files.show(truncate=False)
    except Exception as e2:
        print(f"Cannot access files directly: {e2}")

print("\n" + "=" * 70)
print("\n✅ Gold layer data persisted in HDFS!")
print("\n📍 HDFS Path:")
print("   • hdfs://namenode:9000/lakehouse/gold/gold_daily_kpis_v2")
print("\n🌐 View in HDFS UI: http://localhost:9870/explorer.html#/lakehouse/gold")

📁 HDFS Gold Layer Directory & Data:


🔍 Gold Daily KPIs Table Schema:
+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|        booking_date|     date|   NULL|
|                city|   string|   NULL|
|      total_bookings|   bigint|   NULL|
|  confirmed_bookings|   bigint|   NULL|
|  cancelled_bookings|   bigint|   NULL|
|    pending_bookings|   bigint|   NULL|
|   cancellation_rate|   double|   NULL|
|       total_revenue|   double|   NULL|
| avg_confirmed_price|   double|   NULL|
|   avg_booking_price|   double|   NULL|
|           min_price|   double|   NULL|
|           max_price|   double|   NULL|
|     avg_star_rating|   double|   NULL|
|    unique_customers|   bigint|   NULL|
|        last_updated|timestamp|   NULL|
|      dbt_updated_at|timestamp|   NULL|
|# Partition Infor...|         |       |
|          # col_name|data_type|comment|
|        booking_date|     date|   NULL|
+--------------------+------

## 9. Visualize KPIs

In [ ]:
# Convert to Pandas for visualization
kpis_pd = gold_df.toPandas()

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Total Bookings by City
city_bookings = kpis_pd.groupby('city')['total_bookings'].sum().sort_values(ascending=False)
axes[0, 0].bar(city_bookings.index, city_bookings.values)
axes[0, 0].set_title('Total Bookings by City')
axes[0, 0].set_xlabel('City')
axes[0, 0].set_ylabel('Total Bookings')

# 2. Revenue by City
city_revenue = kpis_pd.groupby('city')['total_revenue'].sum().sort_values(ascending=False)
axes[0, 1].bar(city_revenue.index, city_revenue.values, color='green')
axes[0, 1].set_title('Total Revenue by City')
axes[0, 1].set_xlabel('City')
axes[0, 1].set_ylabel('Revenue')

# 3. Cancellation Rate by City
city_cancel = kpis_pd.groupby('city')['cancellation_rate'].mean().sort_values(ascending=False)
axes[1, 0].bar(city_cancel.index, city_cancel.values, color='red')
axes[1, 0].set_title('Average Cancellation Rate by City')
axes[1, 0].set_xlabel('City')
axes[1, 0].set_ylabel('Cancellation Rate (%)')

# 4. Average Price by City
city_price = kpis_pd.groupby('city')['avg_booking_price'].mean().sort_values(ascending=False)
axes[1, 1].bar(city_price.index, city_price.values, color='orange')
axes[1, 1].set_title('Average Booking Price by City')
axes[1, 1].set_xlabel('City')
axes[1, 1].set_ylabel('Price')

plt.tight_layout()
plt.show()

print("\n✅ KPI visualizations generated!")

## Summary

### Gold Layer Created:
- ✅ **gold_daily_kpis_v2**: Daily KPI aggregations by city
- ✅ **HDFS Storage**: Iceberg format with Snappy Parquet compression
- ✅ **PostgreSQL Storage**: Materialized table for analytics/BI tools
- ✅ **Data Quality**: All tests passed

### Storage Details:
- **HDFS Location**: `hdfs://namenode:9000/lakehouse/gold/gold_daily_kpis_v2`
- **Format**: Apache Iceberg
- **File Format**: Parquet
- **Compression**: Snappy
- **Partitioning**: By booking_date
- **PostgreSQL**: `gold_layer.gold.gold_daily_kpis_postgres` table

### KPIs Calculated:
- ✅ Total bookings per day/city
- ✅ Confirmed/cancelled/pending bookings
- ✅ Cancellation rate
- ✅ Total revenue
- ✅ Average prices (confirmed, all bookings)
- ✅ Min/max prices
- ✅ Average star rating
- ✅ Unique customers

### Data Quality Tests:
- ✅ Unique combination of (booking_date, city)
- ✅ Not null constraints
- ✅ Valid ranges (cancellation_rate 0-100%, revenue >= 0)
- ✅ Booking counts consistency

### Pipeline Complete!
**CSV → Kafka → Bronze (HDFS) → Silver (HDFS) → Gold (HDFS + PostgreSQL)**